# ⚠️ 공통 주의사항

- 행정동 shp 파일은 캠퍼스 내 `[1. 시각화참조파일 > 1. 지역경계shp파일]`에 있음
- 좌표계: **Korea 2000/Central Belt (EPSG:5181)** → WGS84(EPSG:4326)로 변환 필요

- 반출 파일은 집계값만 → 시각화 코드는 집에서 돌리면 됨
- 그림파일(PNG, HTML) 형태 반출 가능 (수치 포함 가능)

In [ ]:
import pandas as pd
import numpy as np

'''
데이터 로드 + `제거
'''
def load_csv(txt_dir):
    df = pd.read_csv('E:/B068. 서울시 연립 다세대 임대 예측시세/2. 파일데이터/'+ txt_dir, sep='|', dtype = str, engine='python')
    df.columns = [str(c).strip().strip("`") for c in df.columns]
    for col in df.columns:
        df[col] = df[col].astype(str).str.strip().str.strip("`")
    return df

df = load_csv('1. 주택기본정보/jtbasicinfo_202208.txt')

In [ ]:
import pandas as pd
import numpy as np
import folium
import geopandas as gpd

export_hvi = pd.read_csv('export_hvi_hdong.csv')

# shp 로드
gdf = gpd.read_file('seoul_hdong.shp')
gdf = gdf.to_crs(epsg=4326)  # folium은 WGS84 기준
gdf = gdf.merge(export_hvi, on='hdong_code', how='left')

# choropleth
m = folium.Map(location=[37.55, 126.98], zoom_start=11)
folium.Choropleth(
    geo_data=gdf,
    data=export_hvi,
    columns=['hdong_code', 'HVI_score'],
    key_on='feature.properties.hdong_code',
    fill_color='YlOrRd',
    legend_name='HVI 점수'
).add_to(m)
m.save('hvi_map.html')

In [ ]:
from folium.plugins import HeatMap

coords = df[['LAT', 'LNG']].dropna().values.tolist()
m = folium.Map(location=[37.55, 126.98], zoom_start=11)
HeatMap(coords, radius=8, blur=10).add_to(m)
m.save('kde_heatmap.html')

In [ ]:
# 모아센터 좌표 리스트
moa_centers = [
    {'name': '○○동 모아센터', 'lat': 37.xx, 'lng': 126.xx},
    ...
]
for c in moa_centers:
    folium.Marker(
        [c['lat'], c['lng']],
        popup=c['name'],
        icon=folium.Icon(color='blue', icon='home')
    ).add_to(m)

# 4. 자치구별 HVI 평균 막대그래프 
: 25개 자치구 단위로 HVI 평균을 내림차순 정렬한 bar chart
* hdong_code 앞 5자리 = 자치구 코드 -> 구단위 집계 가능
* 노원구, 도봉구, 중랑구가 상위 같은 메시지 전달용

In [ ]:
import matplotlib.pyplot as plt

export_hvi['gu_code'] = export_hvi['hdong_code'].str[:5]
gu_avg = export_hvi.groupby('gu_code')['HVI_score'].mean().sort_values(ascending=False)

gu_avg.plot(kind='bar', figsize=(14,5), color='tomato')
plt.title('자치구별 평균 HVI 점수')
plt.tight_layout()
plt.savefig('gu_hvi_bar.png', dpi=150)

# 5. 3개 축 산점도 (노후도 vs 저가 비율, 크기 = 집적도)
> x축: old_ratio, y축: low_ratio_combined, 버블 크기: total_units
> 
- HVI가 높은 동이 왜 높은지 3개 축을 동시에 보여주는 차트
- 발표에서 "이 3가지가 동시에 높은 동을 후보지로 선정" 논리 시각화

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(10,7))
sc = ax.scatter(
    export_hvi['old_ratio'],
    export_hvi['low_ratio_combined'],
    s=export_hvi['total_units'] / 10,  # 버블 크기
    c=export_hvi['HVI_score'],
    cmap='YlOrRd', alpha=0.7
)
plt.colorbar(sc, label='HVI 점수')
ax.set_xlabel('노후도 비율')
ax.set_ylabel('저가 주거지 비율')
ax.set_title('행정동별 HVI 구성 요소 분포')
plt.tight_layout()
plt.savefig('hvi_scatter.png', dpi=150)

## 6. HVI 상위 N개 동 순위표

> 단순하지만 발표에서 "후보지 Top 15" 테이블로 쓸 수 있음
>

In [ ]:
top15 = export_hvi.sort_values('HVI_rank').head(15)[[
    'hdong_name', 'old_ratio', 'low_ratio_combined',
    'total_units', 'HVI_score', 'HVI_grade'
]]
print(top15.to_string(index=False))